# Job Task: Train All Models

> **This notebook runs as a Databricks Job task.**

Task 3 of the retrain pipeline (runs after `ingest_and_validate`):
- Retrieve the feature table created by Task 1 (`feature_engineering`)
- Build a Feature Store training set (joins labels with feature lookups by `customerID`)
- Train Logistic Regression, Random Forest, and Gradient Boosted Trees
- Log all runs to MLflow — models are logged with `fe.log_model()` for lineage tracking
- Pass the best run_id to downstream tasks via task values

In [ ]:
dbutils.widgets.text("catalog", "workshop", "Catalog")
dbutils.widgets.text("schema", "", "Schema (leave blank to derive from current user)")
dbutils.widgets.text("experiment_name", "experiment", "Experiment Name")
dbutils.widgets.text("config_path", "config.yml", "Config Path")
dbutils.widgets.text("source_table", "", "Source Table (for labels; leave blank for default)")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

In [ ]:
%pip install /Volumes/{catalog}/00_shared/wheels/churn_model-0.1.0-py3-none-any.whl databricks-feature-engineering --quiet

In [0]:
dbutils.library.restartPython()

In [ ]:
import yaml
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

catalog         = dbutils.widgets.get("catalog")
schema          = dbutils.widgets.get("schema")
experiment_name = dbutils.widgets.get("experiment_name")
config_path     = dbutils.widgets.get("config_path")
source_table_override = dbutils.widgets.get("source_table")

# Derive schema from the running user when not explicitly provided
if not schema:
    import re as _re
    _user = spark.sql("SELECT current_user()").first()[0]
    schema = _re.sub(r'[^a-z0-9]', '_', _user.split('@')[0].lower()).strip('_')
    print(f"schema derived from current_user: {schema}")

with open(config_path) as f:
    config = yaml.safe_load(f)

# Get the feature table name written by Task 1 (feature_engineering)
feature_table_name = dbutils.jobs.taskValues.get(
    taskKey="feature_engineering",
    key="feature_table_name",
    default=f"{catalog}.{schema}.churn_features",
)
print(f"Feature table : {feature_table_name}")

# Source table provides the labels (customerID + Churn)
source_table = source_table_override if source_table_override else f"{catalog}.`00_shared`.telco_churn"
print(f"Label source  : {source_table}")

# Build the training set: join label_df with feature lookups by customerID
fe = FeatureEngineeringClient()
label_df = spark.table(source_table).select("customerID", config.get("target_column", "Churn"))

training_set = fe.create_training_set(
    df=label_df,
    feature_lookups=[
        FeatureLookup(
            table_name=feature_table_name,
            lookup_key="customerID",
        )
    ],
    label=config.get("target_column", "Churn"),
    exclude_columns=["customerID"],
)
print(f"Training set  : {label_df.count():,} label rows → joined with feature table")

from churn_model.train import run_all_models
results = run_all_models(
    catalog=catalog,
    schema=schema,
    config=config,
    experiment_name=experiment_name,
    fe=fe,
    training_set=training_set,
)

from churn_model.evaluate import get_best_run
best_run_id, best_model_type, best_metrics = get_best_run(
    experiment_name=experiment_name,
    metric="test_f1",
)

print(f"Best model  : {best_model_type}")
print(f"Best run ID : {best_run_id}")
print(f"F1          : {best_metrics['test_f1']:.4f}")

# Pass to downstream tasks
dbutils.jobs.taskValues.set("best_run_id", best_run_id)
dbutils.jobs.taskValues.set("best_model_type", best_model_type)
dbutils.jobs.taskValues.set("best_f1", str(best_metrics["test_f1"]))